<a href="https://colab.research.google.com/github/klorenzoe/hotel-booking-ML-project/blob/feat%2F3%2Ffeature-engineering/hotel_booking_ml.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

# 1 - [Jorge] Recolección y calidad del dato

**Hotel Booking Demand · Universidad de Deusto**

**Objetivos cuando se está realizando la reserva:**

1. Predecir **si se cancelará** (`is_canceled`)
2. Predecir **tarifa media diaria** (`adr`, *average daily rate*: según el paper, la suma de lo facturado por alojamiento
÷ el número de noches de estancia; tabla 1, p. 43)

**Origen de los datos**

- **Paper:** Antonio, de Almeida y Nunes (2019), [*Hotel booking demand datasets*](https://doi.org/10.1016/j.dib.2018.11.126),
  *Data in Brief* 22, 41–49 · CC BY 4.0.
- **Copia usada:** [TidyTuesday 2020-02-11](https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-02-11/hotels.csv) (los dos hoteles en un solo CSV).

## 1.1 Preparar el entorno

In [ ]:
import hashlib
import json
import platform
from datetime import datetime, timezone
from pathlib import Path
from urllib.request import urlretrieve

import numpy as np
import pandas as pd

SEED = 42
np.random.seed(SEED)
pd.set_option("display.max_columns", None)

ROOT = Path.cwd()
RAW_PATH = ROOT / "data" / "raw" / "hotels.csv"
OUT_DIR = ROOT / "outputs" / "01-data-collection-and-quality"
OUT_DIR.mkdir(parents=True, exist_ok=True)

def check(condition, ok, fail):
    """Print ✅ if the condition holds and ⚠️ otherwise."""
    print(f"✅ {ok}" if bool(condition) else f"⚠️ {fail}")


def pct(mask):
    """True values in %."""
    return round(mask.mean() * 100, 1)


def sha256_file(path: Path) -> str:
    """File fingerprint: if a single byte changes, the hash changes."""
    return hashlib.sha256(path.read_bytes()).hexdigest()



## 1.2 ¿El fichero es el correcto?

In [ ]:
SOURCE: dict[str, str] = {
    "name": "hotel-booking-demand",
    "doi": "10.1016/j.dib.2018.11.126",
    "license": "CC BY 4.0",
    "url": "https://raw.githubusercontent.com/rfordatascience/tidytuesday/master/data/2020/2020-02-11/hotels.csv",
    "expected_sha256": "7c2ae42a7353905ea136e5c2287f17c92c5435826598bfbb8491c6f0c7b1fc06",
}

# Expected Values
EXPECTED_ROWS = 119_390  # Paper: Abstract (40,060 + 79,330)
EXPECTED_COLS = 32       # Paper: Abstract (31 variables) + hotel (TidyTuesday)

H1_NAME = "Resort Hotel" # Name by Tidytuesday
H1_BOOKINGS = 40_060     # Paper: Abstract
H1_CANCELED = 11_122     # Paper: Table 3

H2_NAME = "City Hotel"   # Name by Tidytuesday
H2_BOOKINGS = 79_330     # Paper: Abstract
H2_CANCELED = 33_102     # Paper: Table 6

PAPER = pd.DataFrame([
    [H1_NAME, H1_BOOKINGS, H1_CANCELED],
    [H2_NAME, H2_BOOKINGS, H2_CANCELED],
], columns=["hotel", "bookings", "canceled"]).set_index("hotel")


# Download raw data only if missing
if not RAW_PATH.exists():
    RAW_PATH.parent.mkdir(parents=True, exist_ok=True)
    urlretrieve(SOURCE["url"], RAW_PATH)


# Read all csv as text
raw = pd.read_csv(RAW_PATH, dtype=str, keep_default_na=False)
raw_sha256: str = sha256_file(RAW_PATH)

# Only the canceled bookings
raw_canceled = raw[raw["is_canceled"] == "1"]


per_hotel = pd.DataFrame({
    "bookings": raw["hotel"].value_counts(),
    "canceled": raw_canceled["hotel"].value_counts(),
}).reindex(PAPER.index) # same index order as PAPER


paper_counts_ok: bool = per_hotel.equals(PAPER)
hash_match_ok = raw_sha256 == SOURCE["expected_sha256"]
rows_cols_match_ok = raw.shape == (EXPECTED_ROWS, EXPECTED_COLS)

check(paper_counts_ok, "Bookings and cancellations counts match the paper.", "Counts differ from the paper.")
check(hash_match_ok, "Hash matches.", "Hash differs: the file changed.")
check(rows_cols_match_ok, f"{EXPECTED_ROWS} rows x {EXPECTED_COLS} columns.", f"Unexpected shape: {raw.shape}.")
per_hotel.join(PAPER.add_suffix(" (paper)"))

✅ Bookings and cancellations counts match the paper.
✅ Hash matches.
✅ 119390 rows x 32 columns.


,bookings,canceled,bookings (paper),canceled (paper)
hotel,,,,
Resort Hotel,40060,11122,40060,11122
City Hotel,79330,33102,79330,33102


## 1.3 ¿Qué representa una fila y cada columna?

- Cada fila es **una reserva**.
- El `role` dice para qué sirve cada columna:
    - `feature`: posible entrada del modelo.
    - `target_cls` / `target_reg`: lo que predecimos.
    - `forbidden`: está en el fichero, pero se rellena después de reservar: **nunca** puede ser entrada.
    - `outcome`: el desenlace de la reserva: **nunca** puede ser entrada.



In [ ]:
DATA_DICTIONARY = pd.DataFrame([
    ("hotel",                          "categorical", "feature"),
    ("is_canceled",                    "binary",      "target_cls"),
    ("lead_time",                      "count_days",  "feature"),
    ("arrival_date_year",              "date_part",   "feature"),
    ("arrival_date_month",             "date_part",   "feature"),
    ("arrival_date_week_number",       "date_part",   "feature"),
    ("arrival_date_day_of_month",      "date_part",   "feature"),
    ("stays_in_weekend_nights",        "count",       "feature"),
    ("stays_in_week_nights",           "count",       "feature"),
    ("adults",                         "count",       "feature"),
    ("children",                       "count",       "feature"),
    ("babies",                         "count",       "feature"),
    ("meal",                           "categorical", "feature"),
    ("country",                        "categorical", "forbidden"),
    ("market_segment",                 "categorical", "feature"),
    ("distribution_channel",           "categorical", "feature"),
    ("is_repeated_guest",              "binary",      "feature"),
    ("previous_cancellations",         "count",       "feature"),
    ("previous_bookings_not_canceled", "count",       "feature"),
    ("reserved_room_type",             "categorical", "feature"),
    ("assigned_room_type",             "categorical", "forbidden"),
    ("booking_changes",                "count",       "forbidden"),
    ("deposit_type",                   "categorical", "forbidden"),
    ("agent",                          "id_code",     "feature"),
    ("company",                        "id_code",     "feature"),
    ("days_in_waiting_list",           "count_days",  "feature"),
    ("customer_type",                  "categorical", "feature"),
    ("adr",                            "continuous",  "target_reg"),
    ("required_car_parking_spaces",    "count",       "forbidden"),
    ("total_of_special_requests",      "count",       "feature"),
    ("reservation_status",             "categorical", "outcome"),
    ("reservation_status_date",        "date",        "outcome"),
], columns=["column", "analytic_type", "role"])

check(set(DATA_DICTIONARY["column"]) == set(raw.columns), "The dictionary covers all 32 columns.", "Dictionary and file do not match.")
DATA_DICTIONARY["role"].value_counts()

✅ The dictionary covers all 32 columns.


,count
role,
feature,23
forbidden,5
outcome,2
target_cls,1
target_reg,1


**Se crea una copia tipada de los datos.**

- Los datos `raw` no se tocan.

- Se crea `df` como copia **tipada** de `raw` usando los tipos establecidos por `DATA_DICTIONARY`.

- La conversión no puede perder datos: solo los `"NA"` de `children` pueden quedar vacíos.

In [ ]:
# Numeric columns, by their type in the data dictionary
NUMERIC_TYPES = ["binary", "count", "count_days", "continuous"]
NUMERIC_COLS: list[str] = DATA_DICTIONARY.query("analytic_type in @NUMERIC_TYPES")["column"].tolist()

df = raw.copy()

# Text to numbers: anything that is not a number becomes empty
df[NUMERIC_COLS] = df[NUMERIC_COLS].apply(pd.to_numeric, errors="coerce")

# Year + month + day → one arrival date
arrival_text = raw["arrival_date_year"] + "-" + raw["arrival_date_month"] + "-" + raw["arrival_date_day_of_month"]
df["arrival_date"] = pd.to_datetime(arrival_text, format="%Y-%B-%d", errors="coerce")

df["total_nights"] = df["stays_in_weekend_nights"] + df["stays_in_week_nights"]

# Only the "NA" sentinel of children may become empty
children_na = int((raw["children"] == "NA").sum())
empty_per_column = df[NUMERIC_COLS].isna().sum()
empty_columns = empty_per_column[empty_per_column > 0].to_dict()

no_data_lost_ok = empty_columns == {"children": children_na}
arrival_dates_ok = df["arrival_date"].notna().all()

check(no_data_lost_ok, f"Only the {children_na} 'NA' of children became empty.", f"Unexpected empty values: {empty_columns}.")
check(arrival_dates_ok, "Every arrival date is valid.", "Some arrival dates could not be built.")

✅ Only the 4 'NA' of children became empty.
✅ Every arrival date is valid.


## 1.4 ¿Hay duplicados?

**Consideraciones**

- **No existe ID de reserva:** los autores eliminan toda identificación de hotel y cliente por temas de privacidad (*Abstract*, p. 41).

- **Duplicado ambiguo:** una fila repetida puede ser un error o dos reservas parecidas (p. ej., de un grupo).

- **Alcance:** esta sección solo mide; el tratamiento se decide en el 02.

In [ ]:
# Every row that has at least one exact copy (the first one included)
dup_all = raw.duplicated(keep=False)

# Only the extra copies (the first one counts as the original)
dup_extra = raw.duplicated()

pd.Series({
    "extra copies (%)": pct(dup_extra),
    "cancel rate, rows with copies (%)": pct(df.loc[dup_all, "is_canceled"] == 1),
    "cancel rate, unique rows (%)": pct(df.loc[~dup_all, "is_canceled"] == 1),
}, name="value").to_frame()


,value
extra copies (%),26.8
"cancel rate, rows with copies (%)",58.4
"cancel rate, unique rows (%)",26.2


**Conclusiones**

- **Muchas copias:** el `26.8 %` de las filas sobra (`dup_extra`); son copias exactas de otra fila.
- **Las copias se cancelan más:** `is_canceled = 1` en el `58.4 %` de las filas con copias (`dup_all`), frente al
  `26.2 %` de las filas únicas (`unique_rows`): más del doble.
- **Riesgo de fuga:** dos copias tienen el mismo `is_canceled`. Si una cae en *train* y la otra en *test*, el
  modelo se evalúa con una fila que ya vio, y las métricas salen mejores de lo real.
- **Implicación:** las filas de un mismo grupo de `dup_all` deben ir al mismo lado de la partición (se decide en
  el 02).

## 1.5 ¿Qué columnas no se pueden usar?

**Consideraciones**

- **Reglas:**
    - El modelo predice al registrar la reserva.
    - Una columna que se rellena después es **fuga**.

- **Momento del dato:**
    - Cada registro es de la **víspera de la llegada**, no del momento de reservar (*Specifications table*, p. 42).
    - Si un valor cambió entre la reserva y la llegada, solo queda el último; es decir, el valor original al reservar no se puede recuperar.

- **Último estado:** Las siguientes variables reflejan el estado final de la reserva (tabla 1, pp. 43-45):
    - `is_canceled`
    - `reservation_status`
    - `reservation_status_date`
    - `days_in_waiting_list`

- **Prohibidas:** Se conocen después de reservar o son el propio desenlace.

- **Dudosas:** Existen al reservar, pero pueden cambiar antes de la llegada (*§2*, pp. 45-46). Se puede probar con y sin ellas, con una lista distinta por tarea.


In [ ]:
# Filled in after booking, or the outcome itself → never an input
FORBIDDEN: list[str] = [
    "required_car_parking_spaces",  # data: always 0 in canceled bookings → filled in on arrival
    "booking_changes",              # paper (Table 1, p. 43): counts changes until check-in or cancellation
    "assigned_room_type",           # paper (Table 1, p. 43): can differ from the reserved type; data: set at check-in
    "deposit_type",                 # paper (Table 1, p. 43): depends on payments made until arrival or cancellation
    "country",                      # paper (§2, p. 45): often corrected at check-in; data: PRT far more common among canceled
    "reservation_status",           # paper (Table 1, p. 44): final status = the target
    "reservation_status_date",      # paper (Table 1, p. 45): date the final status was set
]

# Known at booking, but stored as of the day before arrival (paper: Specifications table, p. 42).
UNDER_REVIEW: dict[str, list[str]] = {
    "classification": [
        "stays_in_*_nights",           # paper (§2, p. 46): differ between canceled and kept bookings
        "adults / children / babies",  # paper (§2, p. 46): differ between canceled and kept bookings
        "meal",                        # paper (§2, p. 46): differs between canceled and kept bookings
        "days_in_waiting_list",        # paper (Table 1, p. 43): known only once the booking is confirmed
        "total_of_special_requests",   # requests can be added until arrival
        "adr (as an input)",           # paper (Table 1, p. 43): depends on the final stay
    ],
    "regression": [  # nights, guests and meal set the price → allowed
        "days_in_waiting_list",
        "total_of_special_requests",
    ],
}

# FORBIDDEN must match the roles of the data dictionary
DATA_FORBIDDEN = DATA_DICTIONARY.query("role in ['forbidden', 'outcome']")
COLS_FORBIDDEN = DATA_FORBIDDEN["column"]
forbidden_ok = set(FORBIDDEN) == set(COLS_FORBIDDEN)
check(
    condition=forbidden_ok,
    ok="FORBIDDEN matches the data dictionary roles.",
    fail="FORBIDDEN does not match the data dictionary.",
)

✅ FORBIDDEN matches the data dictionary roles.


In [ ]:
# Evidence 1: reservation_status is the target (Canceled or No-Show → is_canceled = 1)
status_canceled = raw["reservation_status"].isin(["Canceled", "No-Show"])
flag_canceled = raw["is_canceled"] == "1"
target_ok = (status_canceled == flag_canceled).all()
n_no_show = (raw["reservation_status"] == "No-Show").sum()
check(
    condition=target_ok,
    ok=f"is_canceled = 1 <---> Canceled or No-Show ({n_no_show:,} no-shows).",
    fail="is_canceled does not match reservation_status.",
)

✅ is_canceled = 1 <---> Canceled or No-Show (1,207 no-shows).


In [ ]:
# Evidence 2: columns filled in after booking differ a lot between arrived and canceled bookings
filled_after_booking = pd.DataFrame({
    "with parking (%)":             df["required_car_parking_spaces"] > 0,
    "assigned room ≠ reserved (%)": raw["assigned_room_type"] != raw["reserved_room_type"],
    "with booking changes (%)":     df["booking_changes"] > 0,
    "Non Refund (%)":               raw["deposit_type"] == "Non Refund",
    "from Portugal (%)":            raw["country"] == "PRT",
})
canceled = df["is_canceled"] == 1
pd.DataFrame({
    "arrived":  filled_after_booking[~canceled].apply(pct),
    "canceled": filled_after_booking[canceled].apply(pct),
})

,arrived,canceled
with parking (%),9.9,0.0
assigned room ≠ reserved (%),18.8,1.8
with booking changes (%),20.3,6.4
Non Refund (%),0.1,32.8
from Portugal (%),28.0,62.2


## 1.6 ¿Cuánta calidad tienen los datos?

## 1.7 ¿A quién representan?

## 1.8 ¿Pasa el lote al notebook 02?

## 1.9 Guardar lo que necesita el notebook 02



---



# 2 - [Mauro] EDA y limpienza

# 3 - [Kathe] Transformación y feature engineering

In [ ]:
#TODO remove this comment